# Two-baseline RL pilot on Colab

Use this notebook with a **remote Colab GPU kernel**, not the local Mac Python.
The validated setup uses one L4, Qwen2.5-0.5B-Instruct and Sokoban PPO.
Vanilla RL and World Model RL (VAGEN text adaptation) each completed 10 updates;
both final evaluation success rates were 0/4. Execution is verified, improvement is not.

Read `docs/handoff.md` before starting. All expensive switches default to `False`.
For an existing runtime, skip extraction and installation; define the helpers,
set RUN_ID, and run the status/plot/case cells. Never relaunch a completed run ID.

Course deliverables: a working framework; loss/reward curves; illustrated case
studies of generated reasoning and actions. Bonus priorities: measured improvement,
interesting RL findings, and additional environments. See `docs/project_requirements.md`.


In [ ]:
import os, sys, json, shlex, platform, subprocess, tarfile
from pathlib import Path
ROOT = Path('/content/RAGEN')
STATE = Path('/content/ragen-jobs')
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
REFERENCE_ENV = Path('/content/ragen-reference')
IS_COLAB = platform.system() == 'Linux' and Path('/content').is_dir()
print('Kernel:', sys.version, sys.executable, 'remote /content:', IS_COLAB)
if IS_COLAB:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
    print(r.stdout or r.stderr)


## 1. Background-job helpers

Run the hardware cell above and this helper cell on every kernel reconnect.
A launch cell returning does not mean the job finished. Use `status(name)` to
read its exit code and log. Colab `/content` is temporary; export results before
releasing the runtime.


In [ ]:
def alive(meta):
    import psutil
    try:
        p=psutil.Process(meta['pid'])
        return abs(p.create_time()-meta['created']) < 0.01 and p.status()!=psutil.STATUS_ZOMBIE
    except psutil.NoSuchProcess:
        return False

def status(name, tail=25):
    path=STATE/f'{name}.json'
    if not path.exists():
        print(name, 'not started'); return
    meta=json.loads(path.read_text()); result=Path(meta['result'])
    print(name, 'exit='+result.read_text().strip() if result.exists() else ('running' if alive(meta) else 'stopped; inspect log'))
    log=Path(meta['log']); print(log)
    if log.exists(): print('\n'.join(log.read_text(errors='replace').splitlines()[-tail:]))

def start(name, script):
    import psutil
    assert IS_COLAB, 'Select a remote Linux Colab kernel'
    assert name.replace('-', '').replace('_','').isalnum()
    STATE.mkdir(exist_ok=True)
    for p in STATE.glob('*.json'):
        old=json.loads(p.read_text())
        if alive(old):
            print('Task still running:', p.stem); status(p.stem); return
    result=STATE/f'{name}.exit'; log=STATE/f'{name}.log'
    if result.exists():
        print('Existing result retained; inspect it or choose a new task name.'); status(name); return
    wrapper='(set -euo pipefail\n'+script+'\n)\nrc=$?\nprintf "%s\n" "$rc" > '+shlex.quote(str(result))+'\nexit "$rc"'
    env=dict(os.environ, TOKENIZERS_PARALLELISM='false', RAY_ADDRESS='local')
    with log.open('w') as stream:
        p=subprocess.Popen(['bash','-c',wrapper],env=env,stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
    meta=dict(pid=p.pid,created=psutil.Process(p.pid).create_time(),result=str(result),log=str(log))
    (STATE/f'{name}.json').write_text(json.dumps(meta))
    print('Started',name,p.pid,log)


## 2. Upload and extract source (new runtime only)

On your local machine, from the repository root:

```bash
git submodule update --init verl
python3 scripts/package_colab.py
```

Upload `outputs/ragen-colab-source.tar.gz` to `/content` using Colab's file panel.
VS Code local edits do not automatically update remote source. Set RUN_EXTRACT
to True below and run it once. The cell refuses to overwrite an existing repo.
The pinned verl commit is `d62da4950573d7a4b7ef2362337952e7ab59e78d`.


In [ ]:
RUN_EXTRACT = False
if RUN_EXTRACT:
    assert IS_COLAB and not ROOT.exists(), 'Repository already exists; sync reviewed source changes manually'
    archive=Path('/content/ragen-colab-source.tar.gz')
    assert archive.is_file(), 'Upload the source archive first'
    with tarfile.open(archive) as t: t.extractall('/content', filter='data')
    assert (ROOT/'scripts/colab/run_two_baselines.py').is_file()
    assert (ROOT/'verl/verl/__init__.py').is_file()


## 3. Install the validated training environment (new runtime only)

Set RUN_REFERENCE_SETUP=True and execute. Then run
`status('reference-setup')` until exit=0. Installation runs in the background;
do not start training before it completes.

This creates `/content/ragen-reference` with Python 3.12, Torch 2.8/cu128,
vLLM 0.11, Transformers 4.57.1 and FlashAttention 2.8.1. The notebook kernel
can use a different Python version. Do not upgrade this stack for Qwen3.5;
its full RL backend is not validated. The installer ends with `pip check`.


In [ ]:
RUN_REFERENCE_SETUP = False
if RUN_REFERENCE_SETUP:
    assert (ROOT/'scripts/colab/setup_reference.sh').is_file()
    start('reference-setup', shlex.join([sys.executable,'-m','pip','install','uv==0.9.5'])+'\n'+
          shlex.join(['bash', str(ROOT/'scripts/colab/setup_reference.sh')]))


## 4. Check GPU imports and environment interaction

Set RUN_REFERENCE_CHECK=True, execute, then use `status('reference-check')`.
This checks Sokoban and FrozenLake reset/step, not PPO learning. WebShop requires
additional Java, product data and search-index setup and is not validated here.
The training runner below also checks both configurations and world-model rewards.


In [ ]:
RUN_REFERENCE_CHECK = False
if RUN_REFERENCE_CHECK:
    py=str(REFERENCE_ENV/'bin/python')
    check="""import torch, vllm, flash_attn
from ragen.env.sokoban.env import SokobanEnv
from ragen.env.frozen_lake.env import FrozenLakeEnv
print(torch.__version__, vllm.__version__, torch.cuda.get_device_name())
for cls in (SokobanEnv, FrozenLakeEnv):
    e=cls(); print(cls.__name__, e.reset(seed=123)); print(e.step(1)); e.close()
"""
    start('reference-check', 'cd '+shlex.quote(str(ROOT))+'\n'+shlex.join([py,'-c',check]))


## 5. Launch the two baselines

For a new run, choose a new RUN_ID (for example `sokoban-two-baselines-v3`),
set STEPS=10 and RUN_COMPARISON=True, then execute once. For viewing the completed
pilot, leave RUN_ID at `sokoban-two-baselines-v2` and RUN_COMPARISON=False.

Runs are sequential on one GPU, each starting from the same base model. Seeds,
batch size and rollout budgets match. World Model adds exact state-prediction
rewards during training only; evaluation uses the original task reward.
The runner refuses to overwrite existing results and stops on failure.


In [ ]:
RUN_ID = 'sokoban-two-baselines-v2'
STEPS = 10
RUN_COMPARISON = False
RESULTS = ROOT/'results/two_baselines'/RUN_ID
if RUN_COMPARISON:
    assert IS_COLAB and (REFERENCE_ENV/'bin/python').is_file()
    if RESULTS.exists():
        print('Existing experiment retained; not restarted:',RESULTS)
        if (RESULTS/'manifest.json').exists(): print((RESULTS/'manifest.json').read_text())
    else:
        start('comparison-'+RUN_ID, 'cd '+shlex.quote(str(ROOT))+'\n'+
              shlex.join([str(REFERENCE_ENV/'bin/python'),'scripts/colab/run_two_baselines.py',
                          '--run-id',RUN_ID,'--steps',str(STEPS)]))


## 6. Monitor progress

Rerun this cell safely. `complete` plus exit code 0 confirms each process finished;
`running` alone is not success. The metrics file records actual completed steps.
If a run fails, read its train.log before attempting a new run ID.


In [ ]:
status('comparison-'+RUN_ID)
manifest=RESULTS/'manifest.json'
if manifest.exists(): print(manifest.read_text())
for name in ('vanilla_rl','world_model_rl'):
    run=RESULTS/name
    metric_file=run/'metrics.jsonl'
    if metric_file.exists():
        records=[json.loads(line) for line in metric_file.read_text().splitlines() if line.strip()]
        print(name,'logged steps:',[r['step'] for r in records])
    log=run/'train.log'
    if log.exists(): print(name,'\n','\n'.join(log.read_text(errors='replace').splitlines()[-8:]))
# First assistant-launched queue's driver log, if present in this runtime.
driver=Path('/content/two-baselines-driver-5.log')
if driver.exists(): print('Driver:',driver.read_text(errors='replace')[-1500:])


## 7. View loss and reward curves

The queue writes curves.png and curves.svg after each baseline finishes.
Set RUN_PLOT=True to regenerate plots from existing JSONL metrics.
Task reward is comparable across methods; optimization reward includes the
World Model bonus and cannot by itself demonstrate better task performance.


In [ ]:
RUN_PLOT = False
if RUN_PLOT:
    subprocess.run([str(REFERENCE_ENV/'bin/python'),str(ROOT/'scripts/colab/plot_pilot.py'),str(RESULTS)],check=True)
if (RESULTS/'curves.png').exists():
    from IPython.display import Image,display
    display(Image(filename=str(RESULTS/'curves.png')))


## 8. Inspect initial/final evaluation cases

Compare the same evaluation seeds before and after training. Inspect generated
reasoning, actions, reward and failure modes. These outputs do not directly reveal
internal model reasoning. Evaluation exports may omit intermediate observations;
train_cases contain structured messages with observations for full-turn inspection.
Select representative examples and explain them in the report; exporting cases
alone does not complete the case-study requirement.


In [ ]:
for name in ('vanilla_rl','world_model_rl'):
    files=sorted((RESULTS/name/'eval_cases').glob('*.jsonl'),key=lambda p:int(p.stem))
    for file in (files[:1]+files[-1:] if len(files)>1 else files):
        sample=json.loads(file.read_text().splitlines()[0])
        print('\n',name,'eval step',file.stem,'score:',sample.get('score'))
        print('INPUT:',sample.get('input','')[:4000])
        print('OUTPUT:',sample.get('output','')[:6000])


## 9. Export results before releasing Colab

Set RUN_EXPORT=True to create an archive in `/content`. Checkpoints are large;
set INCLUDE_CHECKPOINTS=True if you need to resume training. Download the archive
from the Colab file panel and verify it locally. Creating an archive is not a backup
until it has been downloaded or copied to persistent storage.


In [ ]:
RUN_EXPORT = False
INCLUDE_CHECKPOINTS = False
if RUN_EXPORT:
    assert RESULTS.is_dir(), 'Run directory not found'
    suffix = 'full' if INCLUDE_CHECKPOINTS else 'report'
    export = Path('/content') / f'{RUN_ID}-{suffix}.tar.gz'
    assert not export.exists(), 'Existing export retained; choose another name'
    with tarfile.open(export, 'w:gz') as archive:
        for file in sorted(RESULTS.rglob('*')):
            if file.is_file() and (INCLUDE_CHECKPOINTS or 'checkpoints' not in file.relative_to(RESULTS).parts):
                archive.add(file, arcname=str(Path(RUN_ID)/file.relative_to(RESULTS)), recursive=False)
    print('Download this archive:', export, 'bytes:', export.stat().st_size)
